# B 모델 학습과 예측 (README §9)
기존 로컬 연구의 Ridge를 옮긴 **비교용 연구 모델**입니다. 승인된 모델이 아니고, 옛 결과와의 숫자 대조는 아직 하지 않았습니다(`reproduction_status: code_equivalent_only`).
산출물은 `analytics/outputs/model/`, `analytics/outputs/predict/`에 저장됩니다.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / 'analytics' / 'src').exists())
SRC = root / 'analytics' / 'src'
sys.path.insert(0, str(SRC))
import json
import pandas as pd
import train, predict, distribution

## 1. 학습
사전 조건: `build_samples.py`, `features.py`가 만든 `outputs/samples`, `outputs/features`가 있어야 합니다.

In [ ]:
import train as T
from load_gold import load_gold, DEFAULT_SOURCE
from splits import DEFAULT_SAMPLES

samples = pd.read_parquet(DEFAULT_SAMPLES)
scale = T.market_scale_from_gold(load_gold(DEFAULT_SOURCE))
model, assign = T.train_model(samples, scale)          # 학습: train 구간, 보정: calibration 구간. 평가 구간은 사용하지 않음
sample_meta = json.loads((DEFAULT_SAMPLES.with_name('samples_metadata.json')).read_text(encoding='utf8'))
paths = T.write_model(model, assign, scale, T.DEFAULT_OUT, sample_meta)
paths

In [ ]:
card = json.loads(Path(paths['card']).read_text(encoding='utf8'))
{k: card[k] for k in ['model_version', 'status', 'reproduction_status', 'assets_modeled', 'ineligible_assets', 'train_start', 'train_end', 'calibration']}

## 2. 예측 (모델은 다시 학습하지 않음)
지정하지 않은 충격은 0으로 가정하고 `shocks.assumed_zero`에 남깁니다.

In [ ]:
P = predict
m, card, scale_tbl = P.load_model()
ms = pd.read_parquet(P.FEATURE_DIR / 'market_state.parquet')
af = pd.read_parquet(P.FEATURE_DIR / 'asset_features.parquet')
shocks = {'usdkrw_pct': 2.0, 'us10y_bp': 50.0}        # 예: 환율 +2%, 미국 10년 +50bp (WTI, VIX는 0 가정)
long, quant, meta = P.predict(m, ms, af, scale_tbl, shocks, card=card)
P.write_prediction(long, quant, meta, P.DEFAULT_OUT)
quant.round(2)

In [ ]:
{k: meta[k] for k in ['observed', 'shocks', 'warnings']}

`joint_sample_status`가 `ok`이면 같은 `draw_id`는 모든 종목에서 같은 날짜의 잔차입니다(포트폴리오 합산 가능).

In [ ]:
meta['model_estimate']['joint']